# Telecom_api data to Bronze

In [ ]:
api_data = spark.read.json("/Volumes/telecom_catalog/default/landing/Telecom_data/telecom_api_data_2026_06_14_11_25_33.json")

In [ ]:
api_data.display()

## Simulation Strategy: Scaling 1 API Record to 1 Million Records

Instead of calling the telecom API 1 million times (which would be impractical), we:
1. **ADF fetches 1 actual API record** with real schema and baseline timestamp from the telecom API and loads it to landing layer.
2. Before writing to bronze layer we **generate 1 million synthetic records** with randomized values within realistic ranges
3. **Simulate time progression** by incrementing the timestamp by 1 second per record (covers ~11.5 days of data)

This approach creates a production-scale dataset for testing Bronze layer ingestion without overloading the API.

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.types import StringType

# 1. Define your scaling factor (e.g., 1,000,000 records)
TOTAL_RECORDS = 1000000

# 2. Generate base rows
base_df = spark.range(0, TOTAL_RECORDS)

# 3. Generate random values matching your actual API schema
simulated_api_df = (base_df
    # Use the actual API timestamp as baseline and increment by seconds
    .withColumn("timestamp", 
                F.expr("to_timestamp('2026-06-14T11:25:46.718220') + MAKE_INTERVAL(0, 0, 0, 0, 0, 0, id)"))
    
    # Generate random device IDs (like your actual data: DEV_1125)
    .withColumn("device_id",F.concat(F.lit("DEV_"),(F.rand() * 901 + 100).cast("int").cast(StringType())))
    
    # Generate random values matching your actual API metric ranges
    .withColumn("active_sessions", (F.rand() * 50 + 10).cast("long"))
    .withColumn("anomaly_score", F.round(F.rand() * 0.5, 2))
    .withColumn("cache_hit_ratio", F.round(F.lit(0.8) + (F.rand() * 0.19), 2))
    .withColumn("crash_count", F.when(F.rand() > 0.999, 1).otherwise(0).cast("long"))
    .withColumn("disk_read_ops", (F.rand() * 10000 + 5000).cast("long"))
    .withColumn("disk_write_ops", (F.rand() * 5000 + 4000).cast("long"))
    .withColumn("failed_requests", (F.rand() * 10).cast("long"))
    .withColumn("fan_speed_rpm", (F.rand() * 500 + 1500).cast("long"))
    .withColumn("io_wait_time_ms", (F.rand() * 15 + 2).cast("long"))
    .withColumn("latency_ms_p99", (F.rand() * 60 + 20).cast("long"))
    .withColumn("packet_loss_percentage", F.round(F.rand() * 0.08, 4))
    .withColumn("power_usage_watts", (F.rand() * 40 + 90).cast("long"))
    .withColumn("process_count", (F.rand() * 50 + 220).cast("long"))
    .withColumn("queue_length", (F.rand() * 8 + 1).cast("long"))
    .withColumn("reboot_count", F.when(F.rand() > 0.9995, 1).otherwise(0).cast("long"))
    .withColumn("request_count", (F.rand() * 4000 + 2000).cast("long"))
    .withColumn("service_restart_count", F.when(F.rand() > 0.995, 1).otherwise(0).cast("long"))
    .withColumn("tcp_connections", (F.rand() * 200 + 200).cast("long"))
    .withColumn("temperature_celsius", F.round(F.rand() * 25 + 40, 1))
    .withColumn("thread_count", (F.rand() * 300 + 1000).cast("long"))
    .withColumn("unauthorized_access_attempts", F.when(F.rand() > 0.98, 1).otherwise(0).cast("long"))
    .withColumn("uptime_percentage", F.round(F.lit(99.0) + (F.rand() * 0.99), 2))
    
    # Drop the temporary id column
    .drop("id")
)


In [ ]:
simulated_api_df.display()

In [ ]:
(
    simulated_api_df.write
    .format("delta")
    .mode("overwrite") 
    .save("/Volumes/telecom_catalog/default/bronze/API_data")
    
)